# Databricks SQL Fundamentals - Interactive Demo

Welcome! This demo will teach you the fundamentals of Databricks SQL and important concepts you need to know.

---

## What is Databricks SQL?

**Databricks SQL** is a SQL interface built on Apache Spark that provides:
* ANSI SQL compliance with Spark extensions
* Direct access to Unity Catalog tables
* Optimized query execution with Photon engine
* Integration with BI tools and dashboards

**Key Difference from Standard SQL:**
Databricks SQL queries are compiled into Spark execution plans, giving you distributed processing power!

---

## What You'll Learn

1. **SQL Transformations** - SELECT, WHERE, CASE, GROUP BY, ORDER BY
2. **Strict Typing** - Type casting and compatibility gotchas
3. **Store Assignment Policy** - INSERT/CREATE TABLE type enforcement
4. **EXPLAIN Plans** - See how SQL becomes Spark execution
5. **Unity Catalog Integration** - Create and query tables

---

## Demo Scenario

We'll analyze NYC taxi trip data to create hourly summaries - the same analysis from the PySpark lab, but using pure SQL!

## 1. Explore Unity Catalog with SQL

**Unity Catalog** is Databricks' unified governance solution for data and AI assets.

**Three-level namespace:**
* **Catalog** - Top-level container (e.g., `samples`, `main`)
* **Schema** - Database within a catalog (e.g., `nyctaxi`, `default`)
* **Table** - Actual data table (e.g., `trips`)

**Full table reference:** `catalog.schema.table`

Let's explore what's available!

In [0]:
%sql
-- List all catalogs you have access to
SHOW CATALOGS

In [0]:
%sql
-- List all schemas in the samples catalog
SHOW SCHEMAS IN samples

In [0]:
%sql
-- List all tables in the nyctaxi schema
SHOW TABLES IN samples.nyctaxi

In [0]:
%sql
-- See the structure of the trips table
DESCRIBE samples.nyctaxi.trips

In [0]:
%sql
-- Look at sample data
SELECT *
FROM samples.nyctaxi.trips
LIMIT 10

## 2. SQL Transformations

Let's build our analysis step by step.

**Transformations we'll apply:**
1. **SELECT** - Choose specific columns (like PySpark's `select()`)
2. **WHERE** - Filter rows (like PySpark's `filter()`)
3. **CASE WHEN** - Add calculated columns (like PySpark's `withColumn()` + `when()`)
4. **GROUP BY** - Aggregate data (like PySpark's `groupBy()`)
5. **ORDER BY** - Sort results (like PySpark's `orderBy()`)

In [0]:
%sql
-- SELECT: Choose only the columns we need
-- This is equivalent to PySpark's df.select()

SELECT 
  tpep_pickup_datetime,
  fare_amount,
  trip_distance
FROM samples.nyctaxi.trips
LIMIT 10

In [0]:
%sql
-- WHERE: Filter for valid trips only
-- This is equivalent to PySpark's df.filter()

SELECT 
  tpep_pickup_datetime,
  fare_amount,
  trip_distance
FROM samples.nyctaxi.trips
WHERE 
  fare_amount > 0
  AND trip_distance > 0
  AND fare_amount < 500
LIMIT 10

In [0]:
%sql
-- Add calculated columns with CASE WHEN
-- This is equivalent to PySpark's withColumn()

SELECT 
  tpep_pickup_datetime,
  fare_amount,
  trip_distance,
  
  -- Extract hour from timestamp (like PySpark's hour() function)
  HOUR(tpep_pickup_datetime) AS pickup_hour,
  
  -- Categorize revenue (like PySpark's when().otherwise())
  CASE 
    WHEN fare_amount < 10 THEN 'Low'
    WHEN fare_amount <= 30 THEN 'Medium'
    ELSE 'High'
  END AS revenue_category
  
FROM samples.nyctaxi.trips
WHERE 
  fare_amount > 0
  AND trip_distance > 0
  AND fare_amount < 500
LIMIT 10

In [0]:
%sql
-- GROUP BY: Aggregate data by hour
-- This is equivalent to PySpark's groupBy().agg()

SELECT 
  HOUR(tpep_pickup_datetime) AS pickup_hour,
  
  -- Aggregation functions
  COUNT(*) AS trip_count,
  ROUND(SUM(fare_amount), 2) AS total_revenue,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(trip_distance), 2) AS avg_distance
  
FROM samples.nyctaxi.trips
WHERE 
  fare_amount > 0
  AND trip_distance > 0
  AND fare_amount < 500
GROUP BY HOUR(tpep_pickup_datetime)

In [0]:
%sql
-- ORDER BY: Sort results chronologically
-- This is equivalent to PySpark's orderBy()

SELECT 
  HOUR(tpep_pickup_datetime) AS pickup_hour,
  COUNT(*) AS trip_count,
  ROUND(SUM(fare_amount), 2) AS total_revenue,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(trip_distance), 2) AS avg_distance
FROM samples.nyctaxi.trips
WHERE 
  fare_amount > 0
  AND trip_distance > 0
  AND fare_amount < 500
GROUP BY HOUR(tpep_pickup_datetime)
ORDER BY pickup_hour ASC

## 3. Gotcha #1: Strict Typing

**Important:** Databricks SQL has type safety rules, but their behavior **varies by compute environment**!

**What you need to know:**
* In **serverless** compute, Databricks may **implicitly convert** types in some scenarios (e.g., `'5' > 10` works because `'5'` is silently cast to `5`)
* Even when `ansi_mode` is set to `true`, serverless environments can still perform implicit conversions for certain comparisons
* This makes things seem like they "just work" - but it can lead to **silent bugs**

**The real danger isn't always an error - it's getting wrong results without any warning!**

**Common gotchas:**
* String-to-string comparison uses **lexicographic** (dictionary) order, not numeric order
* `'10' > '9'` returns `false` because `'1'` comes before `'9'` alphabetically
* Implicit conversion is **inconsistent** - it works for `'5' > 3` but crashes on `'abc' > 3`
* Mixing types in JOINs can silently drop or duplicate rows

Let's see this in action!

In [0]:
%sql
-- THE SILENT BUG: String comparison uses dictionary order, not numeric order!
-- This query runs without any error... but gives you the WRONG answer.

SELECT 
  '10' > '9'  AS string_comparison,   -- false! ('1' < '9' in dictionary order)
  10 > 9      AS numeric_comparison,   -- true  (correct numeric result)
  '2' > '19'  AS another_trap,         -- true!  ('2' > '1' in dictionary order)
  2 > 19      AS numeric_truth          -- false  (correct numeric result)

-- No error, no warning — just silently wrong results.
-- This is the REAL gotcha with types: not crashes, but invisible bugs!

In [0]:
%sql
-- SOLUTION: Always use explicit CAST when comparing values of different origins

SELECT 
  CAST('10' AS INT) > 9  AS fixed_comparison_1,  -- true (correct!)
  CAST('2' AS INT) > 19  AS fixed_comparison_2,  -- false (correct!)
  CAST('10' AS INT)      AS casted_value,
  9                      AS int_value

-- By casting strings to INT, you get proper numeric comparison.
-- Rule of thumb: if it looks like a number, CAST it as a number before comparing!

In [0]:
%sql
-- REAL-WORLD SCENARIO: What if fare_amount were stored as STRING?
-- Let's simulate this and see the silent bug in a WHERE clause

WITH string_fares AS (
  SELECT 
    CAST(fare_amount AS STRING) AS fare_str,
    fare_amount AS fare_num
  FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
  LIMIT 1000
)

-- String filter: '9.5' > '50' is TRUE because '9' > '5' lexicographically!
-- But numerically 9.5 is NOT greater than 50
SELECT 
  fare_str,
  fare_num,
  fare_str > '50'              AS string_filter,   -- WRONG: uses dictionary order
  CAST(fare_str AS DOUBLE) > 50 AS correct_filter   -- RIGHT: uses numeric order
FROM string_fares
WHERE fare_str > '50' AND fare_num <= 50  -- Rows where string says YES but number says NO
LIMIT 10

-- These are fares UNDER 50 that a string filter would INCORRECTLY include!

In [0]:
%sql
-- IMPLICIT CONVERSION: Inconsistent and unreliable!
-- In serverless compute, some cross-type comparisons work via implicit cast:

SELECT 
  '5' > 3         AS implicit_works,    -- true (serverless casts '5' to 5)
  '100' > 50       AS implicit_works_2,  -- true (serverless casts '100' to 100)
  '5' + 10         AS implicit_addition  -- 15 (serverless casts '5' to 5)

-- But this would CRASH:
-- SELECT 'abc' > 3   -- ERROR: cannot cast 'abc' to numeric
--
-- The problem: implicit conversion is a gamble.
-- It works when the string happens to be a valid number, and crashes when it's not.
-- ALWAYS use explicit CAST() to make your intent clear and your code reliable!

### Common Type Casting Functions

**Casting syntax:**
```sql
CAST(column AS type)
-- or
column::type  -- PostgreSQL-style shorthand
```

**Common types:**
* `INT` / `BIGINT` - Integer numbers
* `DOUBLE` / `DECIMAL(p,s)` - Floating point numbers
* `STRING` - Text
* `BOOLEAN` - True/False
* `DATE` - Date only
* `TIMESTAMP` - Date and time

**Examples:**
```sql
CAST('123' AS INT)                    -- String to integer
CAST(123 AS STRING)                   -- Integer to string
CAST('2024-01-15' AS DATE)           -- String to date
CAST(fare_amount AS DECIMAL(10,2))   -- Specify precision
'123'::INT                            -- Shorthand syntax
```

**Best Practice:**
Always cast explicitly when working with different types - don't rely on implicit conversion!

In [0]:
%sql
-- Check data types in your query results
-- Use DESCRIBE QUERY to see what types your expressions produce

DESCRIBE QUERY (
  SELECT 
    fare_amount,
    CAST(fare_amount AS STRING) AS fare_string,
    CAST(fare_amount AS INT) AS fare_int,
    trip_distance,
    CAST(trip_distance AS DECIMAL(10,2)) AS distance_decimal
  FROM samples.nyctaxi.trips
  LIMIT 1
)

-- This is a great way to verify types BEFORE inserting into a target table!

## 4. Gotcha #2: Store Assignment Policy

**What is Store Assignment Policy?**

Databricks SQL enforces **type checking** when writing data to tables — but not all mismatches are treated equally:
* **Hard failures:** Inserting an incompatible type (e.g., `'three'` into an INT column) raises an error
* **Silent rounding:** Inserting a DECIMAL with extra precision (e.g., `199.999` into `DECIMAL(10,2)`) **silently rounds** to `200.00` — no error, no warning!

**Why does this matter?**
* Hard failures are easy to catch — silent rounding is the real danger
* Your data can change without any indication
* Ensures you must be explicit about precision and scale

**Common scenarios:**
* Inserting STRING into INT column → **ERROR** (good!)
* Inserting higher-precision DECIMAL into lower-precision column → **SILENT ROUNDING** (dangerous!)
* NULL handling in NOT NULL columns → **ERROR** (good!)

**Key Takeaway:** Don't assume all type mismatches will fail. Decimal rounding happens silently — always verify your data after inserts!

Let's see examples!

In [0]:
%sql
-- First, let's create a test table with specific types
-- We'll use this to demonstrate store assignment policy

CREATE OR REPLACE TABLE workspace.default.test_store_assignment (
  id INT,
  name STRING,
  amount DECIMAL(10, 2),
  created_date DATE
)
COMMENT 'Test table for store assignment policy demo'

In [0]:
%sql
-- This INSERT works - types match exactly

INSERT INTO workspace.default.test_store_assignment
VALUES 
  (1, 'Item A', 99.99, '2024-01-15'),
  (2, 'Item B', 149.50, '2024-01-16');

-- Check the data
SELECT * FROM workspace.default.test_store_assignment

In [0]:
%sql
-- This will FAIL - trying to insert STRING into INT column

INSERT INTO workspace.default.test_store_assignment
VALUES 
  ('three', 'Item C', 199.99, '2024-01-17')  -- 'three' is STRING, not INT

-- ERROR: Cannot safely cast 'three':STRING to id:INT

In [0]:
%sql
-- SURPRISE: This does NOT fail!
-- Table expects DECIMAL(10,2) but we're providing 3 decimal places.
-- Databricks SQL silently ROUNDS the value to fit the target precision.
-- 199.999 becomes 200.00 — no error, no warning!

INSERT INTO workspace.default.test_store_assignment
VALUES 
  (3, 'Item D', 199.999, '2024-01-17');  -- 3 decimal places → silently rounded to 200.00

-- Verify: notice amount is 200.00, NOT 199.999
SELECT * FROM workspace.default.test_store_assignment WHERE id = 3

-- LESSON: Store assignment policy catches type CATEGORY mismatches (e.g., STRING → INT)
-- but silently rounds decimals to fit the target scale. This can change your data!

In [0]:
%sql
    
-- Solution: Cast to the correct type

-- WARNING: CAST to DECIMAL still ROUNDS!
-- CAST(199.999 AS DECIMAL(10,2)) → 200.00 (rounded up, same silent rounding!)

-- This works! But note: amount will be 200.00 due to rounding
INSERT INTO workspace.default.test_store_assignment
VALUES 
  (CAST('3' AS INT), 'Item C', CAST(199.999 AS DECIMAL(10,2)), CAST('2024-01-17' AS DATE));

-- To TRUNCATE instead of rounding (keep only the first 2 decimal places):
-- Use FLOOR to chop off extra precision:
--   CAST(FLOOR(199.999 * 100) / 100 AS DECIMAL(10,2))  → 199.99 (no rounding!)
--
-- How it works:
--   199.999 * 100   = 19999.9
--   FLOOR(19999.9)   = 19999
--   19999 / 100      = 199.99

SELECT * FROM workspace.default.test_store_assignment

## 5. SQL is Just an Abstraction for Spark!

**Key Insight:** Databricks SQL queries are compiled into Spark execution plans!

**What does this mean?**
* Your SQL query is translated into Spark operations
* Executed on the same Spark engine as PySpark/Scala
* Benefits from Spark optimizations (Catalyst, Photon)
* Can see the execution plan with `EXPLAIN`

**Why is this important?**
* Understanding the plan helps debug performance issues
* See what Spark is actually doing
* Identify optimization opportunities
* Compare different query approaches

Let's explore the execution plan!

In [0]:
%sql
-- EXPLAIN shows the execution plan for a query
-- Let's start with a simple query

EXPLAIN
SELECT 
  trip_distance,
  fare_amount
FROM samples.nyctaxi.trips
WHERE fare_amount > 50
LIMIT 10

### Understanding EXPLAIN Output

**The execution plan shows:**

1. **Physical Plan** - How Spark will actually execute the query
2. **Operations** - Each step in the execution (scan, filter, project, etc.)
3. **Data Flow** - Bottom-up execution (read from bottom to top)

**Common operations you'll see:**

* **Scan** - Reading data from a table
* **Filter** - WHERE clause conditions
* **Project** - SELECT column selection
* **HashAggregate** - GROUP BY aggregations
* **Sort** - ORDER BY sorting
* **Exchange** - Data shuffle between nodes
* **BroadcastExchange** - Broadcasting small tables

**Reading the plan:**
```
== Physical Plan ==
(3) GlobalLimit 10              ← LIMIT 10
(2) Filter (fare_amount > 50)   ← WHERE clause
(1) Scan table                  ← FROM clause (start here)
```

Read from bottom to top!

In [0]:
%sql
-- EXPLAIN for our aggregation query
-- This shows a more complex plan with shuffles

EXPLAIN
SELECT 
  HOUR(tpep_pickup_datetime) AS pickup_hour,
  COUNT(*) AS trip_count,
  ROUND(SUM(fare_amount), 2) AS total_revenue,
  ROUND(AVG(fare_amount), 2) AS avg_fare
FROM samples.nyctaxi.trips
WHERE 
  fare_amount > 0
  AND trip_distance > 0
GROUP BY HOUR(tpep_pickup_datetime)
ORDER BY pickup_hour

### EXPLAIN Variations

Databricks SQL provides different EXPLAIN modes:

**1. EXPLAIN (default) - Physical Plan**
```sql
EXPLAIN
SELECT * FROM table
```
Shows the actual execution plan.

**2. EXPLAIN EXTENDED - Detailed Information**
```sql
EXPLAIN EXTENDED
SELECT * FROM table
```
Shows parsed logical plan, analyzed plan, optimized plan, and physical plan.

**3. EXPLAIN FORMATTED - Pretty Print**
```sql
EXPLAIN FORMATTED
SELECT * FROM table
```
Shows plan in a more readable tree format.

**4. EXPLAIN COST - Cost-Based Optimization**
```sql
EXPLAIN COST
SELECT * FROM table
```
Shows estimated costs for each operation.

Let's try EXPLAIN EXTENDED!

In [0]:
%sql
-- EXPLAIN EXTENDED shows all optimization stages

EXPLAIN EXTENDED
SELECT 
  HOUR(tpep_pickup_datetime) AS pickup_hour,
  COUNT(*) AS trip_count
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
GROUP BY HOUR(tpep_pickup_datetime)
LIMIT 5

### Key Insights from EXPLAIN

**What to look for in execution plans:**

**1. Scan Operations**
* **ColumnarToRow** - Reading from columnar format (Parquet/Delta)
* **Partition filters** - Pushed down to storage
* **Data skipping** - Using statistics to skip files

**2. Shuffle Operations (Exchange)**
* **HashPartitioning** - Data redistributed by hash of key
* **RoundRobinPartitioning** - Even distribution
* **SinglePartition** - All data to one partition

Shuffles are expensive - minimize them!

**3. Aggregations**
* **HashAggregate** - Hash-based aggregation (fast)
* **Partial** - Pre-aggregation before shuffle
* **Final** - Final aggregation after shuffle

**4. Optimizations**
* **Predicate pushdown** - Filters applied early
* **Column pruning** - Only read needed columns
* **Constant folding** - Evaluate constants at compile time
* **AQE** - Adaptive Query Execution optimizations

**Pro Tip:**
Compare EXPLAIN output for different query approaches to choose the most efficient one!

## 6. Create the Summary Table

Now let's create our final table in Unity Catalog using **CREATE TABLE AS SELECT (CTAS)**.

**CTAS Benefits:**
* Creates table and inserts data in one statement
* Automatically infers schema from query
* Efficient - no separate INSERT needed
* Perfect for ETL pipelines

**Syntax:**
```sql
CREATE OR REPLACE TABLE catalog.schema.table_name AS
SELECT ...
```

In [0]:
%sql
-- Create the taxi hourly summary table
-- This combines all our transformations!

CREATE OR REPLACE TABLE workspace.default.taxi_hourly_summary AS
SELECT 
  HOUR(tpep_pickup_datetime) AS pickup_hour,
  COUNT(*) AS trip_count,
  ROUND(SUM(fare_amount), 2) AS total_revenue,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(trip_distance), 2) AS avg_distance
FROM samples.nyctaxi.trips
WHERE 
  fare_amount > 0
  AND trip_distance > 0
  AND fare_amount < 500
GROUP BY HOUR(tpep_pickup_datetime)
ORDER BY pickup_hour

In [0]:
%sql
-- Verify the table was created successfully

SELECT * FROM workspace.default.taxi_hourly_summary
ORDER BY pickup_hour

In [0]:
%sql
-- Check the schema that was inferred from our query

DESCRIBE workspace.default.taxi_hourly_summary

In [0]:
%sql
-- See detailed table information

DESCRIBE EXTENDED workspace.default.taxi_hourly_summary

## SQL vs PySpark: Same Result, Different Syntax

Let's compare how we achieved the same result in both languages:

### **Complete Comparison**

| Operation | SQL | PySpark |
|-----------|-----|----------|
| **Read data** | `SELECT * FROM table` | `spark.read.table("table")` |
| **Select columns** | `SELECT col1, col2` | `.select("col1", "col2")` |
| **Filter rows** | `WHERE condition` | `.filter(condition)` |
| **Add column** | `CASE WHEN ... END AS col` | `.withColumn("col", when()...)` |
| **Extract hour** | `HOUR(timestamp)` | `hour("timestamp")` |
| **Aggregate** | `GROUP BY col` | `.groupBy("col").agg(...)` |
| **Count** | `COUNT(*)` | `count("*")` |
| **Sum** | `SUM(col)` | `sum("col")` |
| **Average** | `AVG(col)` | `avg("col")` |
| **Round** | `ROUND(col, 2)` | `round(col, 2)` |
| **Sort** | `ORDER BY col` | `.orderBy("col")` |
| **Create table** | `CREATE TABLE AS SELECT` | `.write.saveAsTable()` |
| **Explain plan** | `EXPLAIN query` | `df.explain()` |

### **Key Insight:**

Both SQL and PySpark compile to the **same Spark execution plan**! Choose based on:
* **SQL** - Familiar syntax, great for analytics, BI tool integration
* **PySpark** - Programmatic control, complex logic, Python ecosystem

## Quick Reference Guide

### **Exploring Unity Catalog**
```sql
SHOW CATALOGS
SHOW SCHEMAS IN catalog_name
SHOW TABLES IN catalog.schema
DESCRIBE catalog.schema.table
DESCRIBE EXTENDED catalog.schema.table
```

### **Type Casting**
```sql
CAST(column AS INT)
CAST(column AS DECIMAL(10,2))
CAST(column AS DATE)
CAST(column AS TIMESTAMP)
column::INT  -- Shorthand
```

### **Aggregations**
```sql
COUNT(*), COUNT(DISTINCT col)
SUM(col), AVG(col)
MIN(col), MAX(col)
ROUND(col, decimals)
```

### **Date/Time Functions**
```sql
HOUR(timestamp), DAY(date), MONTH(date), YEAR(date)
DATE_TRUNC('hour', timestamp)
DATE_ADD(date, days)
DATEDIFF(date1, date2)
CURRENT_DATE(), CURRENT_TIMESTAMP()
```

### **Conditional Logic**
```sql
CASE 
  WHEN condition1 THEN value1
  WHEN condition2 THEN value2
  ELSE default_value
END
```

### **Table Operations**
```sql
CREATE OR REPLACE TABLE catalog.schema.table AS SELECT ...
INSERT INTO catalog.schema.table VALUES (...)
INSERT INTO catalog.schema.table SELECT ...
DROP TABLE catalog.schema.table
```

### **Debugging**
```sql
EXPLAIN query
EXPLAIN EXTENDED query
DESCRIBE QUERY (SELECT ...)
```

## Congratulations!

You've completed the Databricks SQL Fundamentals demo!

### **What You Learned:**

* **SQL Transformations** - SELECT, WHERE, CASE, GROUP BY, ORDER BY  
* **Strict Typing** - Explicit casting required for type safety  
* **Store Assignment Policy** - Type enforcement in INSERT/CREATE  
* **EXPLAIN Plans** - SQL queries compile to Spark execution plans  
* **Unity Catalog** - Modern data governance and table management  
* **SQL = Spark** - Same engine, different syntax  

---

### **Key Takeaways:**

1. **Databricks SQL is Spark** - Your queries run on the same distributed engine
2. **Type safety matters** - Always cast explicitly to avoid errors
3. **EXPLAIN is your friend** - Use it to understand and optimize queries
4. **Unity Catalog is the standard** - Use three-level namespace
5. **SQL and PySpark are equivalent** - Choose based on your use case

---

### **Next Steps:**

* Practice writing complex SQL queries
* Explore advanced SQL functions (window functions, CTEs, etc.)
* Learn about Delta Lake features (MERGE, time travel, etc.)
* Study query optimization techniques
* Build SQL-based data pipelines

---

### **Resources:**

* [Databricks SQL Documentation](https://docs.databricks.com/sql/)
* [SQL Language Reference](https://docs.databricks.com/sql/language-manual/)
* [Unity Catalog Guide](https://docs.databricks.com/data-governance/unity-catalog/)
* [Query Optimization](https://docs.databricks.com/optimizations/)